In [1]:
import pymaid
import navis
import pandas as pd
import numpy as np
import zarr
import docker
import tensorstore as ts
import os
import json
from ac_segmentation.neurotorch.datasets.dataset import open_ZarrTensor

In [3]:
###import neurons into CATMAID from SWC file
def import_neurons(swc_path, server, api_token, project_id):
    #Load instance and set project
    rm = pymaid.CatmaidInstance(server=server,
                            api_token=api_token)
    rm.project_id = project_id

    #load swc fata and reformat
    ns = navis.read_swc(swc_path)
    n_table = ns.nodes[['node_id', 'parent_id', 'x', 'y', 'z', 'radius']]
    subtrees = ns.subtrees
    
    for i in range(len(subtrees)):
        nodes = subtrees[i]
        neu = n_table[n_table['node_id'].isin(nodes)]
        neu = navis.TreeNeuron(neu)
        resp = pymaid.upload_neuron(neu)

In [ ]:
#import_neurons(swc_path="/ACdata/Users/connorl/skeletons/For_Wan-Qing/skeletons_10_with_orientation.swc", #server="http://bigkahuna:4554/", api_token='7deede658f4bc1f53ee6993fc28ec26fbbe5838f', project_id = 16 )

In [33]:
###convert zarr to n5
def zarr_to_n5(zarr_path, out_path, out_name = "Data_Out", chunks=(64,64,64), cutout=None):
    #open zarr
    arr = open_ZarrTensor(zarr_path)
    if cutout != None:
        x1,x2,y1,y2,z1,z2 = cutout
        arr = arr[0,0,x1:x2,y1:y2,z1:z2].transpose().read().result()
    else:
        arr = arr[0,0,:,:,:].transpose().read().result()

    #create n5
    store = zarr.N5Store(os.path.join(out_path, out_name + '.n5'))
    root = zarr.group(store=store)
    z = root.zeros('group/' + zarr_path[-2], shape=arr.shape, chunks=chunks, dtype=arr.dtype, compressor=None)
    z[:] = arr

In [29]:
#zarr_to_n5(zarr_path='/ACdata/Users/kevin/ispim_ome_zarr/H17_x55_S39a_230808_highres/H17_x55_S39a_230808_highres.zarr/highres_Pos90/1/', out_path='/ACdata/Users/connorl/N5_Files/', chunks=(64,64,64), cutout=[12000,16000,0,288,0,288])

In [34]:
###convert zarr to precomputed
def zarr_to_precomputed(zarr_path, out_path, out_name = "Data_Out", chunks=(64,64,64), cutout=None):
    #open zarr
    arr = open_ZarrTensor(zarr_path)
    if cutout != None:
        x1,x2,y1,y2,z1,z2 = cutout
        arr = arr[0,:,x1:x2,y1:y2,z1:z2].read().result()
        arr = np.transpose(arr, (1, 2, 3, 0))
    else:
        arr = arr[0,:,:,:,:].read().result()
        arr = np.transpose(arr, (1, 2, 3, 0))

    #get resolution
    r_path = os.path.join(os.path.dirname(os.path.dirname(zarr_path)), ".zattrs")
    res = json.loads(open(r_path, "r").read())['multiscales'][0]['datasets'][int(zarr_path[-2])]['coordinateTransformations'][0]['scale'][2:]
    
    #create precomputed tensor
    pre_comp = ts.open(
         {
             "driver": "neuroglancer_precomputed",
             "kvstore": {
                 "driver": "file",
                 "path": os.path.join(out_path, out_name),
             },
             "scale_metadata": {
                 "resolution": res,
                 "chunk_size": list(chunks),
                 "encoding": "raw",
                 "key": zarr_path[-2]
             }
         },
         create=True,
         dtype=arr.dtype,
         domain=ts.IndexDomain(
             shape=list(arr.shape),
         )).result()

    pre_comp.write(arr).result()

In [35]:
#zarr_to_precomputed(zarr_path='/ACdata/Users/kevin/ispim_ome_zarr/H17_x55_S39a_230808_highres/H17_x55_S39a_230808_highres.zarr/highres_Pos90/1/', out_path='/ACdata/Users/connorl/N5_Files/', chunks=(64,64,64), cutout=[12000,16000,0,288,0,288])

In [56]:
def import_Zarr_ToCATMAID(zarr_path, out_path, container_id, out_name='Data_Out', project='NewProject', stack='NewStack', 
                          chunks=(64,64,64), translation=(0,0,0), cutout=None, format='precomputed'):

    if format=='precomputed':
        zarr_to_precomputed(zarr_path=zarr_path, out_path=out_path, out_name=out_name, chunks=(64,64,64), cutout=cutout)
        tile_type = 14
        type_path = out_name + zarr_path[-2] + "/0_1_2"

    elif format=='n5':
        zarr_to_n5(zarr_path=zarr_path, out_path=out_path, out_name=out_name, chunks=(64,64,64), cutout=cutout)
        tile_type = 11
        type_path = out_name+ ".n5/group/" + zarr_path[-2] +  "/0_1_2"

    else:
        raise ValueError("Format must be N5 or precomputed")

    #get resolution and shape
    shape = json.load(open(os.path.join(out_path,"zarr_data.n5/group/dataset/attributes.json")))['dimensions']
    r_path = os.path.join(os.path.dirname(os.path.dirname(zarr_path)), ".zattrs")
    res = json.loads(open(r_path, "r").read())['multiscales'][0]['datasets'][int(zarr_path[-2])]['coordinateTransformations'][0]['scale'][2:]
    
    #create project data json
    stack_file = [{
      "project": {
        "title": project,
        "stacks": [{
          "title": stack,
          "dimension": str(tuple(shape)),
          "mirrors": [{
            "fileextension": "n5",
            "position": 0,
            "tile_source_type": tile_type,
            "tile_height": shape[1],
            "tile_width": shape[0],
            "title": "Example tiles",
            "url": os.path.join("http://bigkahuna.corp.alleninstitute.org" + out_path, type_path)
          }],
          "resolution": str(tuple(res)),
          "translation": str(translation),
          "downsample_factors": ["(1,1,1)"]
        }]
      }
    }]
    
    json_fpath = os.path.join(out_path, "data.json")
    with open(json_fpath, 'w') as f:
        json.dump(stack_file, f)

    #copy json file to container
    copy_string = json_fpath + " " + container_id + ":" + "/home/django/projects/data.json"
    ! docker cp $copy_string
    
    #open docker container
    client = docker.from_env()
    container = client.containers.get(container_id)
    
    #import json to container
    container.exec_run('python3 manage.py catmaid_import_projects --input data.json')

    #cleanup
    container.exec_run('rm input data.json')
    client.close()
    os.remove(json_fpath)

In [57]:
#import_Zarr_ToCATMAID(zarr_path="/ACdata/Users/kevin/ispim_ome_zarr/H17_x55_S39a_230808_highres/H17_x55_S39a_230808_highres.zarr/highres_Pos90/1/", out_path="/ACdata/Users/connorl/N5_Files/", container_id='9024a7f6d341', cutout=[12000,16000,0,288,0,288], project='Project5', stack='Stack5')